# Regime ML: Production Integration

## Goal
Understand how regime_ml.py integrates with model.py and how to use it in your portfolio system.

In [ ]:
print('''
════════════════════════════════════════════════════════════
PRODUCTION API: compute_regime_ml()
════════════════════════════════════════════════════════════

From regime_ml.py:

def compute_regime_ml(
    con: duckdb.DuckDBPyConnection,
    model_path: Path = MODEL_PATH,
) -> tuple[str, dict[str, float]]:
    """
    Public API for model.py integration.
    
    Returns:
      (regime_label, {sector: multiplier})
    
    Falls back to ('UNKNOWN', {}) if model file does not exist or fails.
    """

Usage in model.py:
──────────────────
    from scripts.scoring.regime_ml import compute_regime_ml
    
    regime, sector_mults = compute_regime_ml(con)
    
    if regime != 'UNKNOWN':
        # Apply ML-based sector multipliers
        for sector, mult in sector_mults.items():
            portfolio[sector] *= mult
    else:
        # Fall back to rule-based multipliers
        portfolio = apply_rule_based_multipliers(portfolio)
''')

## Sector Multipliers

How regimes translate to portfolio tilts

In [ ]:
print('''
SECTOR MULTIPLIERS BY REGIME
════════════════════════════════════════════════════════════

Each regime has empirical sector multipliers computed from historical data.
Multiplier formula: 1.0 + 4.0 × (excess_return)
Clipped to [0.70, 1.50] range

EXPANSION (SPY +7% over next 3m):
  Technology:         1.30x  ▲ boost growth
  Consumer Disc:      1.20x  ▲ cyclical strength
  Financials:         1.10x  ▲ light tilt
  Utilities:          0.80x  ▼ reduce defensive
  Consumer Staples:   0.75x  ▼ reduce defensive

CONTRACTION (SPY -5% over next 3m):
  Utilities:          1.35x  ▲ overweight defensive
  Consumer Staples:   1.25x  ▲ stable revenue
  Healthcare:         1.15x  ▲ defensive
  Technology:         0.65x  ▼ underweight growth
  Discretionary:      0.70x  ▼ underweight cyclical

RECOVERY (SPY 0-7%, bouncing from weakness):
  Energy:             1.25x  ▲ beaten-down sector
  Tech:               1.15x  ▲ partial recovery
  Staples:            0.90x  ▼ reduce ultra-defensive
  Utilities:          0.85x  ▼ reduce defensive

LATE_CYCLE (SPY flat -5% to +7%):
  All sectors:        ~1.00x  balanced allocation
  Watch for:          transition signals

How to Apply:
─────────────
Starting allocation: {SPY: 0.40, BND: 0.30, ...}
Adjusted = {sector: weight × multiplier[regime][sector]}
Renormalize so weights sum to 1.0
''')

## Production Workflow

In [ ]:
print('''
PRODUCTION WORKFLOW
════════════════════════════════════════════════════════════

1. SETUP (Once)
   ──────────────────────────────────────────────────────
   python3 scripts/scoring/regime_ml.py --train
   
   This:
   - Loads full DuckDB history
   - Builds all 20+ features
   - Trains GradientBoostingClassifier
   - Computes empirical sector multipliers
   - Saves to projects/investing/inputs/regime_model.pkl
   
   Runtime: ~5-10 minutes

2. INFERENCE (Daily, as part of rebalance)
   ──────────────────────────────────────────────────────
   from scripts.scoring.regime_ml import compute_regime_ml
   
   regime, sector_mults = compute_regime_ml(con)
   
   if regime != 'UNKNOWN':
       # Apply multipliers
       for sector, mult in sector_mults.items():
           portfolio[sector] *= mult
       portfolio /= portfolio.sum()  # Renormalize
   else:
       # Fallback to rules-based
       portfolio = apply_rule_based_multipliers(portfolio)
   
   Runtime: <1 second (just inference)

3. MONITORING (Monthly)
   ──────────────────────────────────────────────────────
   python3 scripts/scoring/regime_ml.py --backtest
   
   This:
   - Runs walk-forward validation
   - Reports OOS accuracy
   - Tracks model drift
   - Alert if accuracy < 55%
   
   Runtime: ~5 minutes

4. RETRAINING (Quarterly or if accuracy drops)
   ──────────────────────────────────────────────────────
   python3 scripts/scoring/regime_ml.py --train
   
   - Re-fits on all new data collected
   - Updates sector multipliers
   - Replaces regime_model.pkl
   
   Runtime: ~5-10 minutes
''')

## Fallback Behavior

What happens if the model fails?

In [ ]:
print('''
GRACEFUL DEGRADATION
════════════════════════════════════════════════════════════

The compute_regime_ml() function is designed to fail safely:

try:
    # Load model
    with open(model_path, 'rb') as f:
        bundle = pickle.load(f)
    
    # Build features from DuckDB
    feat = build_features(con)
    
    # Make prediction
    regime, probs = predict_current(feat, model, ...)
    mults = bundle['sector_multipliers'].get(regime, {})
    
    return regime, mults

except Exception:
    # Fallback: return UNKNOWN
    return 'UNKNOWN', {}

In model.py:
────────────
regime, mults = compute_regime_ml(con)

if regime == 'UNKNOWN':
    # Use rule-based multipliers (which already work well)
    mults = get_rule_based_multipliers()
    
    # Log the failure (for debugging)
    logger.warning('Regime ML failed, using rules-based')

# Apply multipliers
portfolio = apply_multipliers(portfolio, mults)

Benefits:
─────────
✓ Portfolio never breaks (always has multipliers)
✓ Graceful degradation (rules-based is good enough)
✓ No cascading failures
✓ System keeps running

Why This Matters:
─────────────────
In production, robustness > perfect accuracy.
A model that works 95% of the time with fallback
is better than a model that works 99% but crashes
when it fails.
''')

## Integration Checklist

In [ ]:
print('''
INTEGRATION CHECKLIST
════════════════════════════════════════════════════════════

☐ 1. Import the API in model.py
      from scripts.scoring.regime_ml import compute_regime_ml

☐ 2. Call it in your portfolio optimization
      regime, sector_mults = compute_regime_ml(con)

☐ 3. Handle UNKNOWN regime
      if regime == 'UNKNOWN':
          sector_mults = fallback_multipliers()

☐ 4. Apply multipliers to base allocation
      for sector in portfolio:
          portfolio[sector] *= sector_mults.get(sector, 1.0)

☐ 5. Renormalize weights
      portfolio /= portfolio.sum()

☐ 6. Set up monitoring
      - Monthly: python3 scripts/scoring/regime_ml.py --backtest
      - Alert if accuracy drops below 55%

☐ 7. Set up retraining
      - Quarterly: python3 scripts/scoring/regime_ml.py --train
      - After major market events (VIX spikes, etc.)

☐ 8. Add logging
      logger.info(f'Regime: {regime}, confidence: {max(probs):.1%}')

☐ 9. Test fallback behavior
      Temporarily move regime_model.pkl to verify fallback works

☐ 10. Document the system
       Add regime signal to portfolio documentation
''')

## Interview Story

How to explain this system to a hiring manager

In [ ]:
print('''
YOUR INTERVIEW STORY
════════════════════════════════════════════════════════════

"I built a machine learning regime detector for our portfolio system.
Rather than hand-coded rules, it learns from 20+ engineered features—
price momentum, market breadth, credit spreads—to predict one of 4
market regimes.

The model achieves ~65% accuracy on walk-forward validated test data,
significantly outperforming random guessing (25%) and beating our
rule-based approach.

In production, we use it to dynamically adjust portfolio sector weights:
- EXPANSION regimes: overweight growth sectors (+30%)
- CONTRACTION regimes: overweight defensive sectors (-30%)
- RECOVERY: tactical tilts in beaten-down sectors

The system integrates via a clean API: compute_regime_ml(con) returns
the predicted regime and sector multipliers. If the model fails for any
reason, we gracefully fall back to rule-based multipliers, ensuring
the portfolio never breaks.

What I learned from this project:
- Regime classification is about consistency, not perfect accuracy
- Walk-forward validation is the only honest way to evaluate time-series models
- Production systems need graceful degradation and fallback behavior
- A simple rule-based system with a good fallback beats a complex model without one

[Show Phase 4 feature importance chart and Phase 6 walk-forward results]
"

Key talking points:
──────────────────
✓ Feature engineering (economic intuition)
✓ Machine learning (GradientBoosting)
✓ Time-series validation (walk-forward)
✓ Production design (graceful fallback)
✓ Business impact (portfolio tilting)
''')

## Summary

You now understand:
- The complete regime_ml.py system from training to production
- How it integrates with model.py
- How to handle failures gracefully
- How to monitor and retrain in production

You're ready to:
1. Train the model: `python3 scripts/scoring/regime_ml.py --train`
2. Integrate with model.py
3. Deploy to production